# Teams with `SelectorGroupChat`

`RoundRobinGroupChat` rotates speakers in a fixed order. **`SelectorGroupChat` is smarter** — an LLM looks at the conversation so far and picks whichever agent is best suited to speak next.

Use it when:
- The right next speaker depends on the situation (e.g. *was that a code question or a planning question?*).
- You don't want every agent to chime in on every turn.
- You have specialists with non-overlapping skills.

We'll build a tiny **planner / coder / reviewer** team and let the selector route turns.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

## Define the specialists

Each agent has a `description` — that's what the selector reads to decide who should speak. Make descriptions **concrete and non-overlapping**, otherwise the selector flips a coin.

In [ ]:
planner = AssistantAgent(
    name="planner",
    model_client=model_client,
    description="Breaks the user's request into a short numbered plan. Speaks first.",
    system_message="Write a 3-step numbered plan for the task. No code.",
)

coder = AssistantAgent(
    name="coder",
    model_client=model_client,
    description="Writes Python code that follows the plan.",
    system_message="Write a single Python snippet that implements the plan. No prose.",
)

reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    description="Reviews the code and says APPROVE or one concrete fix.",
    system_message="If the code is correct, reply only APPROVE. Otherwise give one short fix.",
)

## Build the team

`SelectorGroupChat` needs a `model_client` of its own — that's the LLM that picks the next speaker. We also pass `allow_repeated_speaker=False` so the same agent doesn't talk twice in a row.

In [ ]:
team = SelectorGroupChat(
    participants=[planner, coder, reviewer],
    model_client=model_client,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
    allow_repeated_speaker=False,
)

await Console(team.run_stream(
    task="Write a function that returns the nth Fibonacci number."
))

Watch the trace: the selector usually goes **planner → coder → reviewer**, but it may loop back to the coder if the reviewer asks for a fix. That routing is the whole point of `SelectorGroupChat`.

## Customizing how the selector decides

Pass `selector_prompt="..."` to override the default selection prompt. The string can reference `{roles}`, `{participants}`, and `{history}`.

In [ ]:
custom_prompt = (
    "Read the conversation. Pick exactly one of {participants} to speak next.\n"
    "Roles:\n{roles}\n\n"
    "Conversation:\n{history}\n\n"
    "Rule: planner speaks first; after any code, the reviewer must speak next.\n"
    "Reply with only the agent name."
)

team2 = SelectorGroupChat(
    participants=[planner, coder, reviewer],
    model_client=model_client,
    selector_prompt=custom_prompt,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)

await Console(team2.run_stream(task="Write a function that checks if a number is prime."))

## Recap

- **`RoundRobinGroupChat`** — fixed rotation. Predictable, dumb.
- **`SelectorGroupChat`** — LLM-routed turns. Picks the right specialist based on `description`.
- Good descriptions matter more than clever prompts.
- Override `selector_prompt` when you need a hard rule the LLM keeps breaking.

In [ ]:
await model_client.close()